# PROJET MODELISATION GEOMETRIQUE

## SUJET

**Surfacique : nuage de points**

De façon similaire au TP sur les maillages, il sera demandé de pouvoir retrouver une tangente et un rayon en chaque point, ainsi qu’un point sur la surface définie par le nuage de points.

In [24]:
# Imports
import numpy as np
import plotly.graph_objects as go
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from utils import *

# Paramètres
NUM_POINTS = 200            # Nombre de points
NUM_NEIGHBORS = 5           # Nombre de voisin par point
GENERATION_TYPE = 'regular' # Le type de repartition des points

### Générer le nuage de points

In [25]:
def generate_spherical_points(radius, num_points, distribution):
    if distribution == 'regular':
        # Generating regularly distributed points on the surface of a sphere
        indices = np.arange(0, num_points, dtype=float) + 0.5
        phi = np.arccos(1 - 2*indices/num_points)
        theta = np.pi * (1 + 5**0.5) * indices
        x, y, z = radius * np.cos(theta) * np.sin(phi), radius * np.sin(theta) * np.sin(phi), radius * np.cos(phi)
        points = np.column_stack((x, y, z))
    elif distribution == 'random':
        # Generating randomly distributed points on the surface of a sphere
        u = np.random.rand(num_points)
        v = np.random.rand(num_points)
        theta = 2 * np.pi * u
        phi = np.arccos(2 * v - 1)
        x = radius * np.sin(phi) * np.cos(theta)
        y = radius * np.sin(phi) * np.sin(theta)
        z = radius * np.cos(phi)
        points = np.column_stack((x, y, z))
        
    return points

# Generate regularly distributed points on the surface of a sphere
points = generate_spherical_points(5, NUM_POINTS, GENERATION_TYPE)

In [26]:
# Create a 3D figure with Plotly
fig = go.Figure()
fig.update_layout(scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'))

# Add points to the figure
fig.add_trace(go.Scatter3d(x=points[:, 0], y=points[:, 1], z=points[:, 2], mode='markers',
                           marker=dict(color='magenta', size=3), showlegend=False))

fig.show()

### Obtenir les voisins

In [27]:
def find_direct_neighbors(points, num_neighbors):
    nbrs = NearestNeighbors(n_neighbors=num_neighbors + 1, algorithm='auto').fit(points)
    distances, indices = nbrs.kneighbors(points)
    return indices

direct_neighbors = find_direct_neighbors(points, NUM_NEIGHBORS)

In [28]:
# Add connections between neighbors
for i, point in enumerate(points):
    for neighbor_index in direct_neighbors[i]:
        neighbor_point = points[neighbor_index]
        # Add a line trace between the point and its neighbor
        fig.add_trace(go.Scatter3d(x=[point[0], neighbor_point[0]],
                                    y=[point[1], neighbor_point[1]],
                                    z=[point[2], neighbor_point[2]],
                                    mode='lines',
                                    line=dict(color='blue', width=2),
                                    opacity=0.5,
                                    showlegend=False))

fig.show()

### Obtenir les k plus proches voisins

In [29]:
def get_k_near_neighbors(neighbors_indices, k):
    k_near_neighbors = [set() for _ in range(NUM_POINTS)]  # Initialize an array of empty sets
    
    # Add the point itself to its own neighbors
    for point in range(NUM_POINTS):
        k_near_neighbors[point].add(point)
    
    # Iterate through k connections
    for _ in range(k):
        for point in range(NUM_POINTS):
            for neighbor in neighbors_indices[point]:
                k_near_neighbors[point].update(neighbors_indices[neighbor])  # Add neighbors of neighbors
    
    # Remove the point itself from the set of neighbors
    for point in range(NUM_POINTS):
        k_near_neighbors[point].remove(point)
    
    return k_near_neighbors

k_near_neighbors = get_k_near_neighbors(direct_neighbors, 2)

In [30]:
fig.add_trace(go.Scatter3d(x=[points[0, 0]], y=[points[0, 1]], z=[points[0, 2]], mode='markers',
                            marker=dict(color='red', size=5), showlegend=False))

for neighbor_index in k_near_neighbors[0]:
    neighbor_point = points[neighbor_index]
    fig.add_trace(go.Scatter3d(x=[neighbor_point[0]],
                                y=[neighbor_point[1]],
                                z=[neighbor_point[2]],
                                mode='markers',
                                marker=dict(color='orange', size=5),
                                showlegend=False))

fig.show()


### Calculer les normales sortantes

In [31]:
def get_normal(points, neighbors):
    normals = []
    for p in range(len(points)):
        # Extract point coordinates and its neighbors' coordinates
        neighbor_points = points[list(neighbors[p])]
        
        # Perform PCA
        pca = PCA(n_components=3)
        pca.fit(neighbor_points)
        
        # Get the normal vector from the first principal component
        normal_vector = pca.components_[2]
        if np.linalg.norm(points[p] + normal_vector) < np.linalg.norm(points[p]) :
            normal_vector =  - normal_vector
        normals.append(normal_vector)
    
    return normals

In [32]:
normals = get_normal(points, k_near_neighbors)


fig = go.Figure()

for i, point in enumerate(points):
    fig.add_trace(go.Scatter3d(x=[point[0]],
                                y=[point[1]],
                                z=[point[2]],
                                mode='markers',
                                marker=dict(color='magenta', size=3),
                                name=f'Point {i+1}'))

for point,normal in zip(points,normals):
    fig.add_trace(go.Scatter3d(x=[point[0], point[0] + normal[0]],
                                y=[point[1], point[1] + normal[1]],
                                z=[point[2], point[2] + normal[2]],
                                mode='lines',
                                line=dict(color='orange', width=2),
                                showlegend=False))

fig.update_layout(scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
                    showlegend=True)
fig.show()


### Trouver les rayons

In [33]:
def get_rayons(points, direct_neighbors):
    rayons = []
    for i, voisins in enumerate(direct_neighbors):
        distances = [np.linalg.norm(points[i] - points[j]) for j in voisins]
        mean_distance = np.mean(distances)
        rayons.append(mean_distance / 2.0)
    return rayons

rayons = get_rayons(points, direct_neighbors)

In [34]:
for point, rayon, normal in zip(points, rayons, normals):
    add_circle_v2(point, rayon, normal, fig)

fig.update_layout(scene=dict(xaxis=dict(title="X"),
                             yaxis=dict(title="Y"),
                             zaxis=dict(title="Z")),
                  showlegend=False)

#fig.show()

In [35]:
def add_circle_v2(point, radius, normal, fig):
    x_center, y_center, z_center = point

    # Normalizing the normal vector
    normal = np.array(normal) / np.linalg.norm(normal)

    # Create an orthogonal vector to the normal vector
    T1 = np.array([0, -normal[2], normal[1]])
    T1 /= np.linalg.norm(T1)

    # Create another orthogonal vector to the normal vector and the first orthogonal vector
    T2 = np.cross(normal, T1)

    # Generate circle points using parametric equations
    theta = np.linspace(0, 2*np.pi, 100)
    x_circle = x_center + radius * (np.cos(theta) * T1[0] + np.sin(theta) * T2[0])
    y_circle = y_center + radius * (np.cos(theta) * T1[1] + np.sin(theta) * T2[1])
    z_circle = z_center + radius * (np.cos(theta) * T1[2] + np.sin(theta) * T2[2])

    # Add circle trace to the figure
    fig.add_trace(go.Scatter3d(
        x=x_circle, y=y_circle, z=z_circle,
        mode='lines',
        line=dict(color='blue', width=2)
    ))

In [36]:
fig = fig = go.Figure()

add_circle_v2([2,2,2], 1, [1,1,1], fig)

fig.update_layout(scene=dict(xaxis=dict(title="X"),
                             yaxis=dict(title="Y"),
                             zaxis=dict(title="Z")),
                  showlegend=False)

fig.show()